# 5. SupportSense — Business Impact & Validation
## Objective
Translate the offline experiment into a cautious business decision. This notebook summarizes the benchmark, compares variants, explores assumed support-handling savings, and documents what evidence is still needed before rollout.

All operational inputs below are editable assumptions—not company facts.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rag_core import SupportKnowledgeBase, baseline_answer, rag_answer, evaluate_answer
import time

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
docs = json.loads((ROOT / "data" / "company_docs.json").read_text(encoding="utf-8"))
questions = json.loads((ROOT / "data" / "evaluation_questions.json").read_text(encoding="utf-8"))
kb = SupportKnowledgeBase(docs)
records = []
for q in questions:
    retrieved = kb.retrieve(q["question"], k=3)
    for variant in ["A_baseline", "B_rag"]:
        start = time.perf_counter()
        result = baseline_answer(q["question"], retrieved) if variant == "A_baseline" else rag_answer(q, retrieved)
        records.append(evaluate_answer(q, result, retrieved, (time.perf_counter()-start)*1000, variant))
results = pd.DataFrame(records)
summary = results.groupby("variant")[["retrieval_hit","grounded_answer_proxy","citation_coverage",
                                      "correct_abstention","unsupported_answer_proxy",
                                      "unnecessary_abstention","estimated_handle_seconds"]].mean()
display(summary.round(3))

## Paired question-level comparison
Because both variants are evaluated on the same questions, compare them question by question. For a real experiment, the primary outcome and analysis method should be chosen before inspecting results.

In [ ]:
wide = results.pivot(index="question_id", columns="variant", values="grounded_answer_proxy")
wide["B_minus_A"] = wide["B_rag"] - wide["A_baseline"]
display(wide)
print("Mean paired proxy difference (B - A):", round(wide["B_minus_A"].mean(), 3))

# Bootstrap interval for the mean paired difference; descriptive only for this small synthetic set.
rng = np.random.default_rng(42)
diffs = wide["B_minus_A"].dropna().to_numpy()
if len(diffs):
    boot = np.array([rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(5000)])
    print("Bootstrap 95% interval:", tuple(np.round(np.quantile(boot, [0.025, 0.975]), 3)))

## Scenario calculator
Change the assumptions below to explore potential time savings. This is not a measured impact estimate: the time values in the benchmark are illustrative scenario inputs.

In [ ]:
monthly_tickets = 5000
baseline_handle_minutes = 6.0
assumed_assisted_handle_minutes = 4.5
eligible_share = 0.30
adoption_share = 0.60
agent_cost_per_hour = 18.0  # currency units per hour; illustrative

assisted_tickets = monthly_tickets * eligible_share * adoption_share
minutes_saved = max(0, baseline_handle_minutes - assumed_assisted_handle_minutes)
monthly_hours_saved = assisted_tickets * minutes_saved / 60
monthly_capacity_value = monthly_hours_saved * agent_cost_per_hour

scenario = pd.DataFrame([{
    "monthly_tickets_assumed": monthly_tickets,
    "eligible_share_assumed": eligible_share,
    "adoption_share_assumed": adoption_share,
    "assisted_tickets_estimated": round(assisted_tickets),
    "minutes_saved_per_assisted_ticket_assumed": minutes_saved,
    "hours_saved_per_month_scenario": round(monthly_hours_saved, 1),
    "capacity_value_per_month_scenario": round(monthly_capacity_value, 2)
}])
display(scenario)

## Recommendation and validation gate
Do not recommend a broad rollout from this synthetic offline benchmark alone. Before launch:

1. Build a larger, representative, human-reviewed test set with answerable and unanswerable questions.
2. Compare the baseline and treatment using a pre-registered primary metric, such as human-rated resolution quality or correct grounded answer rate.
3. Add citation correctness, unsafe/unsupported answers, abstention quality, latency, and escalation rate as guardrails.
4. Run a randomized pilot with enough traffic to estimate uncertainty; segment by issue type and customer tier where appropriate.
5. Measure actual agent handle time and customer outcomes before claiming cost savings.
6. Review privacy, access controls, data retention, and auditability with the business owner.